# Benchmark: Python puro vs Numba

Compara el tiempo por paso Monte Carlo del barrido de Metrópolis en Python puro
y compilado con `@njit` (`montecarlo.barrido_metropolis`), y el paso híbrido
Metrópolis + Wolff. Debe estar en la misma carpeta que los módulos del proyecto.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import parametros
from red import enlaces_por_anillos, construir_libreta, L_minimo
from montecarlo import sembrar, barrido_metropolis, barrido_temperaturas

enl, _ = enlaces_por_anillos(parametros.cell, parametros.posiciones,
                             parametros.anillos, parametros.tol, parametros.factor_J)
nat = len(parametros.posiciones)
print("L mínimo:", L_minimo(enl))

## 1. Versión en Python puro (mismo algoritmo, sin compilar)

In [ ]:
def barrido_metropolis_py(spins, vec, Jv, nvec, beta):
    N = spins.size
    for _ in range(N):
        i = np.random.randint(N)
        h = 0.0
        for c in range(nvec[i]):
            h += Jv[i, c] * spins[vec[i, c]]
        dE = 2.0 * spins[i] * h
        if dE <= 0.0 or np.random.random() < np.exp(-beta * dE):
            spins[i] = -spins[i]

## 2. Medición
La primera llamada a una función `@njit` incluye la compilación: se descarta.

In [ ]:
L_vals = [6, 12, 18, 24]
beta = 1.0 / (parametros.k_B * 300.0)
filas = []
for L in L_vals:
    vec, Jv, nvec = construir_libreta(L, L, nat, enl)
    N = len(nvec)
    spins = np.random.choice([-1, 1], size=N).astype(np.int64)
    barrido_metropolis(spins, vec, Jv, nvec, beta)          # compilación
    n_py, n_nb = 5, 500
    t0 = time.perf_counter()
    for _ in range(n_py):
        barrido_metropolis_py(spins, vec, Jv, nvec, beta)
    t_py = (time.perf_counter() - t0) / n_py
    t0 = time.perf_counter()
    for _ in range(n_nb):
        barrido_metropolis(spins, vec, Jv, nvec, beta)
    t_nb = (time.perf_counter() - t0) / n_nb
    filas.append((L, N, t_py, t_nb))
    print(f"L={L:3d} N={N:5d}  Python={t_py*1e3:9.3f} ms  Numba={t_nb*1e3:8.4f} ms  x{t_py/t_nb:6.1f}")

In [ ]:
Ns = [f[1] for f in filas]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.loglog(Ns, [f[2]*1e3 for f in filas], "o-", label="Python puro")
a1.loglog(Ns, [f[3]*1e3 for f in filas], "s-", label="Numba")
a1.set_xlabel("N espines"); a1.set_ylabel("ms por barrido"); a1.legend()
a2.plot(Ns, [f[2]/f[3] for f in filas], "d-")
a2.set_xlabel("N espines"); a2.set_ylabel("aceleración")
plt.tight_layout(); plt.savefig("resultados/benchmark_numba.png", dpi=150); plt.show()